# Pipeline 3: Claim-Verified Multi-Hop RAG
### Claim-Verified Multi-Hop RAG — DATASCI 266, Summer 2026

This notebook implements the final intervention pipeline using a calibrated hop-level verifier and verifier-triggered correction actions.



### Why this design
Pipeline 2 established two facts:
1. Multi-hop chaining significantly improves EM/F1 over the single-pass baseline.
2. Unsupported intermediate claims still correlate with lower final-answer accuracy.

The goal here is to preserve Pipeline 2's multi-hop gains while reducing hallucination propagation by applying a targeted correction step when a hop is flagged as unsupported.

### Intervention policy used in this notebook
- Run normal multi-hop generation for hop 1 and hop 2.
- Verify each hop answer against retrieved evidence using the calibrated verifier.
- **If a hop is UNSUPPORTED**, run one rescue attempt:
  - re-retrieve with larger top-k,
  - regenerate with a stricter grounding prompt,
  - re-verify and keep the rescue answer only if it improves support.
- **UNANSWERABLE is not force-corrected by default** in the primary run to keep comparison with Pipeline 2 fair and isolate unsupported-claim intervention effects.



In [2]:
# Install all dependencies
# Same packages as Pipeline 2, plus datasets/transformers/torch for verifier-guided runs.
# rank_bm25              : fast BM25 implementation
# sentence-transformers  : pretrained bi-encoder for dense embeddings
# faiss-cpu              : vector similarity search
# openai                 : GPT-4o-mini for generation
# tqdm                   : progress bars
# datasets               : HotpotQA loading
# transformers / torch   : local NLI verifier inference

%pip install rank_bm25 sentence-transformers faiss-cpu openai tqdm datasets transformers torch --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 68.0 MB/s eta 0:00:00


In [3]:
import time
import os
import re
import string
import json
import pickle
import numpy as np
from collections import Counter
from tqdm.auto import tqdm

import torch
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import faiss
from openai import OpenAI
from datasets import load_dataset
from google.colab import drive, userdata

In [4]:
# OpenAI API key
# In Colab: store your key via Secrets as OPENAI_API_KEY
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
client = OpenAI()

## 1. Load Shared Artifacts from Drive

To keep comparisons fair and reproducible, this notebook reuses the same saved assets as Pipelines 1 and 2:
- corpus and FAISS index from Pipeline 1,
- shared 2,000-example subset indices,
- saved decompositions from Pipeline 2,
- Pipeline 1 and Pipeline 2 result files for final comparison.

Only generation for this new intervention pipeline is recomputed.

In [5]:
# Load shared artifacts from Google Drive

# Mount Google Drive to access files saved by Pipelines 1 and 2
drive.mount("/content/drive")
SAVE_DIR = "/content/drive/MyDrive/datasci266_rag/"

# Load serialized corpus dictionary
with open(f"{SAVE_DIR}corpus.pkl", "rb") as f:
    saved = pickle.load(f)

# corpus_docs[i] = full text of paragraph i
# corpus_titles[i] = Wikipedia article title of paragraph i
# val_gold_indices[j] = set of corpus indices that are gold paragraphs for val example j
corpus_docs = saved["docs"]
corpus_titles = saved["titles"]
val_gold_indices = saved["val_gold_indices"]

# Load pre-built FAISS index
index = faiss.read_index(f"{SAVE_DIR}faiss.index")

# Shared subset + saved decomposition outputs from Pipeline 2
eval_indices_shared = np.load(f"{SAVE_DIR}eval_indices_shared.npy")
with open(f"{SAVE_DIR}decompositions.json") as f:
    decompositions = json.load(f)

# Pipeline outputs used for locked config + comparison tables
with open(f"{SAVE_DIR}naive_rag_full_results.json") as f:
    p1_output = json.load(f)
with open(f"{SAVE_DIR}multihop_rag_results.json") as f:
    p2_output = json.load(f)

print(f"Corpus loaded: {len(corpus_docs):,} passages")
print(f"FAISS vectors: {index.ntotal:,}")
print(f"Shared subset size: {len(eval_indices_shared):,}")
print(f"Saved decompositions: {len(decompositions):,}")

Mounted at /content/drive
Corpus loaded: 54,391 passages
FAISS vectors: 54,391
Shared subset size: 2,000
Saved decompositions: 2,000


In [6]:
# Rebuild BM25 index
# BM25 is rebuilt from corpus_docs to keep tokenization behavior identical to prior notebooks.
print("Building BM25...")

# Convert each paragraph to lowercase tokens — matches how queries will be tokenized
tokenized_corpus = [doc.lower().split() for doc in corpus_docs]

# Initialize BM25Okapi: computes term frequencies and document lengths across the corpus
bm25 = BM25Okapi(tokenized_corpus)

# Load sentence-transformer (for query encoding only)
EMBED_MODEL = "all-MiniLM-L6-v2"
embedder = SentenceTransformer(EMBED_MODEL)

# Inherit locked retrieval config from Pipeline 1 for fair comparisons
BEST_MODE = p1_output["best_mode"]
BEST_TOP_K = p1_output["best_top_k"]

print("BM25 ready.")
print(f"Embedder loaded: {EMBED_MODEL}")
print(f"Inherited retrieval config: mode={BEST_MODE!r}, top_k={BEST_TOP_K}")

Building BM25...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

BM25 ready.
Embedder loaded: all-MiniLM-L6-v2
Inherited retrieval config: mode='hybrid', top_k=10


In [7]:
# Hybrid Retrieval via Reciprocal Rank Fusion (RRF)
# Identical retrieval utilities to prior notebooks.

def bm25_retrieve(query: str, top_n: int = 100) -> list[int]:
    """Return top_n corpus indices ranked by BM25 score."""
    # Preprocess query text exactly like the corpus tokenization step
    tokens = query.lower().split()
    # Calculate BM25 matching scores for all documents in the index
    scores = bm25.get_scores(tokens)
    # Sort indices based on score (argsort yields low-to-high, [::-1] reverses it to high-to-low)
    ranked = np.argsort(scores)[::-1]
    # Slice out the requested number of top candidates and convert to a standard Python list
    return ranked[:top_n].tolist()


def dense_retrieve(query: str, top_n: int = 100) -> list[int]:
    """Return top_n corpus indices ranked by cosine similarity (FAISS)."""
    # Convert query into a normalized vector embedding matching the corpus format
    q_emb = embedder.encode([query], normalize_embeddings=True, convert_to_numpy=True)
    # Run kNN vector search inside the FAISS index
    _, indices = index.search(q_emb, top_n)
    # Extract row from batch wrapper and convert to list
    return indices[0].tolist()


def reciprocal_rank_fusion(ranked_lists: list[list[int]], k: int = 60) -> list[int]:
    """
    Combine multiple ranked lists with RRF.

    For each document d appearing at rank r in a list,
    its RRF score = sum over lists of 1 / (k + r).
    Returns document indices sorted by descending RRF score.
    """
    # Key = document index, value = cumulative RRF score
    scores: dict[int, float] = {}
    for ranked in ranked_lists:
        for rank, doc_idx in enumerate(ranked, start=1):
            scores[doc_idx] = scores.get(doc_idx, 0.0) + 1.0 / (k + rank)
    return sorted(scores, key=scores.__getitem__, reverse=True)


def hybrid_retrieve(query: str, top_k: int = 10, candidate_n: int = 100) -> list[int]:
    """
    Full hybrid retrieval pipeline.

    1. BM25 retrieves `candidate_n` candidates.
    2. Dense retrieves `candidate_n` candidates.
    3. RRF merges and re-ranks both lists.
    4. Return the top_k corpus indices.
    """
    # 1) Keyword retrieval
    bm25_results = bm25_retrieve(query, top_n=candidate_n)
    # 2) Dense retrieval
    dense_results = dense_retrieve(query, top_n=candidate_n)
    # 3) Fusion
    fused = reciprocal_rank_fusion([bm25_results, dense_results])
    # 4) Final top-k
    return fused[:top_k]


# Retrieval mode switcher
# Single entry point so downstream code is retriever-agnostic.
def retrieve(query: str, top_k: int = 10, mode: str = "hybrid") -> list[int]:
    """
    Unified retrieval interface.

    mode="bm25"   sparse lexical only
    mode="dense"  semantic only
    mode="hybrid" RRF fusion of both
    """
    if mode == "bm25":
        return bm25_retrieve(query, top_n=top_k)
    elif mode == "dense":
        return dense_retrieve(query, top_n=top_k)
    elif mode == "hybrid":
        return hybrid_retrieve(query, top_k=top_k)
    else:
        raise ValueError(f"Unknown retrieval mode: {mode!r}. Choose 'bm25', 'dense', or 'hybrid'.")


def build_context_string(doc_indices: list[int]) -> str:
    """Format retrieved paragraphs into a numbered context block for the prompt."""
    parts = []
    # Loop over document indices, formatting into a readable reference block for the LLM
    for i, idx in enumerate(doc_indices, 1):
        parts.append(f"[{i}] {corpus_titles[idx]}\n{corpus_docs[idx]}")
    # Join documents with double line breaks for clean structural spacing
    return "\n\n".join(parts)

In [8]:
# Normalization and evaluation metrics (same pattern as prior notebooks)
def normalize_answer(s: str) -> str:
    """Lowercase, remove punctuation, articles, and extra whitespace."""
    # Uses regex to replace standalone articles ('a', 'an', 'the') with a space
    def remove_articles(text):
        return re.sub(r"\b(a|an|the)\b", " ", text)

    # Strips out duplicate internal spaces, tabs, and newlines
    def white_space_fix(text):
        return " ".join(text.split())

    # Drops punctuation to avoid penalizing commas/periods
    def remove_punc(text):
        exclude = set(string.punctuation)
        return "".join(ch for ch in text if ch not in exclude)

    # Standardizes the string by running all cleaning steps in sequence
    return white_space_fix(remove_articles(remove_punc(s.lower())))


# Returns 1 for identical cleaned-string match, 0 otherwise
def exact_match(prediction: str, gold: str) -> int:
    """1 if normalized prediction exactly matches normalized gold answer."""
    return int(normalize_answer(prediction) == normalize_answer(gold))


# Break down cleaned prediction and gold strings into token lists
def token_f1(prediction: str, gold: str) -> float:
    """
    Token-level F1 score.
    Measures partial credit when answers are multi-word phrases.
    """
    pred_tokens = normalize_answer(prediction).split()
    gold_tokens = normalize_answer(gold).split()

    # Finds overlapping tokens via multiset intersection
    common = Counter(pred_tokens) & Counter(gold_tokens)
    num_same = sum(common.values())

    # Quick exit for zero overlap
    if num_same == 0:
        return 0.0

    precision = num_same / len(pred_tokens)
    recall = num_same / len(gold_tokens)
    return 2 * precision * recall / (precision + recall)


# Checks whether any gold paragraph appears in retrieved set
def retrieval_recall_at_k(retrieved_indices: list[int], gold_indices: set[int]) -> int:
    """
    1 if any gold paragraph appears in retrieved set, else 0.
    Soft recall metric.
    """
    return int(len(set(retrieved_indices) & gold_indices) > 0)


# Returns 1 only if all required gold paragraphs are present
def both_gold_retrieved(retrieved_indices: list[int], gold_indices: set[int]) -> int:
    """1 if ALL gold paragraphs appear in retrieved set (strict recall)."""
    return int(gold_indices.issubset(set(retrieved_indices)))


def summarize_results(results: list[dict]) -> dict:
    """Aggregate result records into mean metrics."""
    n = len(results)
    return {
        "n": n,
        "EM": sum(r["em"] for r in results) / n,
        "F1": sum(r["f1"] for r in results) / n,
        "Recall@k (soft)": sum(r["recall_soft"] for r in results) / n,
        "Recall@k (strict)": sum(r["recall_strict"] for r in results) / n,
        "Abstention rate": sum(r["abstained"] for r in results) / n,
        "Hop1 recall": sum(r["hop1_recall"] for r in results) / n,
        "Hop2 recall": sum(r["hop2_recall"] for r in results) / n,
        # Pipeline 3 tracking: post-verifier hop grounding rates
        "Hop1 ungrounded": sum(r["hop1_ungrounded"] for r in results) / n,
        "Hop2 ungrounded": sum(r["hop2_ungrounded"] for r in results) / n,
    }

## 2. Generation + Verifier Utilities

This section the calibrated verifier protocol from Pipeline 2:
- QA-to-claim conversion,
- per-passage entailment scoring,
- max-entailment aggregation,
- separate `UNANSWERABLE` state.

The key addition is intervention logic that attempts one rescue step when a hop is unsupported.

In [9]:
# API retry wrapper
# Wraps API calls with exponential backoff on rate limit errors.
# This prevents a single rate limit blip from crashing long runs.
from openai import RateLimitError

def call_with_retry(fn, max_retries: int = 5):
    """
    Call fn() and retry with exponential backoff if RateLimitError is raised.
    Raises if all retries are exhausted.
    """
    for attempt in range(max_retries):
        try:
            return fn()
        except RateLimitError:
            if attempt == max_retries - 1:
                raise
            wait_seconds = 10 * (2 ** attempt)
            print(f"\nRate limit hit. Waiting {wait_seconds}s...")
            time.sleep(wait_seconds)


# Prompt templates
# Keep prompts concise and consistent with prior pipelines.
SYSTEM_PROMPT = """You are a precise question-answering assistant.
Answer the question using ONLY the provided context passages.
Give a short, direct answer (a name, date, number, or brief phrase).
If the context does not contain enough information to answer, respond with exactly: UNANSWERABLE"""

# Rescue prompt is slightly stricter after verifier-triggered retries.
RESCUE_SYSTEM_PROMPT = """You are a careful evidence-grounded QA assistant.
Use ONLY the provided passages.
If uncertain, prefer UNANSWERABLE over guessing.
Give a short direct answer only."""


def generate_answer(question: str, context: str, model: str = "gpt-4o-mini", system_prompt: str = SYSTEM_PROMPT) -> str:
    """
    Call GPT-4o-mini with question + context.
    temperature=0 for deterministic outputs.
    """
    user_message = f"""Context passages:
{context}

Question: {question}

Answer:"""
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_message},
        ],
        temperature=0,
        max_tokens=64,
    )
    return response.choices[0].message.content.strip()


def get_decomposition(question: str) -> list[str]:
    # Reuse saved decompositions from Pipeline 2 for fair apples-to-apples comparison.
    return decompositions.get(question, [question, question])

In [10]:
# Load selected verifier model (from Pipeline 2 calibration output if available)
BEST_VERIFIER = "cross-encoder/nli-deberta-v3-base"
MAX_PASSAGES_VERIFY = 3
ENTAILMENT_THRESHOLD = 0.50

# If Pipeline 2 grounding output exists, inherit selected verifier for consistency.
try:
    with open(f"{SAVE_DIR}multihop_rag_grounding_results.json") as f:
        g_meta = json.load(f)
    BEST_VERIFIER = g_meta.get("nli_model", BEST_VERIFIER)
except Exception:
    pass

# Load model once
# Load selected Sequence Classification weights and put model in evaluation mode
print(f"Loading verifier: {BEST_VERIFIER}")
nli_tokenizer = AutoTokenizer.from_pretrained(BEST_VERIFIER)
nli_model = AutoModelForSequenceClassification.from_pretrained(BEST_VERIFIER)
nli_model.eval()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
nli_model = nli_model.to(DEVICE)

# Dynamically construct label map lookup arrays from model metadata properties
label_to_idx = {v.lower(): k for k, v in nli_model.config.id2label.items()}
ENTAILMENT_IDX = label_to_idx.get("entailment", 1)
print(f"Verifier loaded on {DEVICE}")

# Text flattener that transforms a question-answer string pair into
# a singular, flattened declarative claim statement for NLI validation.
def qa_to_claim(question: str, answer: str) -> str:
    q = question.strip()
    a = answer.strip()
    if not q:
        return a
    q_no_qmark = q[:-1] if q.endswith("?") else q
    # Match standard English structural interrogative prefix strings
    m = re.match(r"^(Who|What|Where|When|Which|Whose|How many|How much)\b", q_no_qmark, flags=re.I)
    # Swap out interrogative keywords with the concrete text answer string
    if m:
        claim = re.sub(r"^(Who|What|Where|When|Which|Whose|How many|How much)\b", a, q_no_qmark, flags=re.I)
        return claim if claim.endswith(".") else claim + "."
    return f"Question: {q} Answer: {a}."


# Computes fine-grained sentence-level entailment probabilities across retrieved
# passages to assess whether a generated answer is grounded in factual evidence.
def verify_answer(question: str, answer: str, context_indices: list[int], max_passages: int = MAX_PASSAGES_VERIFY) -> dict:
    # Keep abstentions as a separate verifier state.
    if "UNANSWERABLE" in answer.upper():
        return {
            "supported": None,
            "state": "UNANSWERABLE",
            "label": "UNANSWERABLE",
            "entailment_score": 1.0,
            "claim": qa_to_claim(question, answer),
            "passage_scores": [],
            "passage_labels": [],
        }
    # If zero reference context indices exist, flag immediately as unsupported
    if not context_indices:
        return {
            "supported": False,
            "state": "UNSUPPORTED",
            "label": "NEUTRAL",
            "entailment_score": 0.0,
            "claim": qa_to_claim(question, answer),
            "passage_scores": [],
            "passage_labels": [],
        }

    # Synthesize the declarative claim target string
    claim = qa_to_claim(question, answer)
    passage_scores = []
    passage_labels = []

    # Iterate through top K evidence passages
    for idx in context_indices[:max_passages]:
        premise = corpus_docs[idx]
        # Format the dual sequence inputs into localized tensor arrays
        inputs = nli_tokenizer(
            premise,
            claim,
            return_tensors="pt",
            truncation=True,
            max_length=512,
            padding=True,
        ).to(DEVICE)
        # Deactivate gradient tape recording loops to optimize inference runtime
        with torch.no_grad():
            logits = nli_model(**inputs).logits
        # Extract normalized probability scores across vocabulary classification targets
        probs = torch.softmax(logits, dim=-1)[0]
        pred_idx = probs.argmax().item()
        # Record tracking metrics for individual sequence passes
        passage_labels.append(nli_model.config.id2label[pred_idx].upper())
        passage_scores.append(probs[ENTAILMENT_IDX].item())

    # Aggregate passage scores by selecting the maximum single-context entailment probability
    max_ent = max(passage_scores) if passage_scores else 0.0
    # Apply calibrated decision criteria to check if factual alignment satisfies thresholds
    is_supported = max_ent >= ENTAILMENT_THRESHOLD

    return {
        "supported": is_supported,
        "state": "SUPPORTED" if is_supported else "UNSUPPORTED",
        "label": "ENTAILMENT" if is_supported else "UNSUPPORTED",
        "entailment_score": max_ent,
        "claim": claim,
        "passage_scores": passage_scores,
        "passage_labels": passage_labels,
        "threshold": ENTAILMENT_THRESHOLD,
    }

Loading verifier: cross-encoder/nli-deberta-v3-base


config.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.35k [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 2.46MB            

spm.model: downloading bytes:           |  0.00B            

...

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  738MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Verifier loaded on cpu


In [11]:
# Intervention helpers
INTERVENE_UNSUPPORTED = True
INTERVENE_UNANSWERABLE = False   # Keep False in primary comparison for fairness vs P2
RESCUE_TOP_K = 20                # Larger retrieval budget during rescue

# Executes a singular reasoning layer step (hop) containing retrieval, answer generation,
# and a local NLI grounding verification pass to intercept potential hallucinations.
def run_hop_with_verification(sub_question: str, retrieval_query: str, top_k: int, retrieval_mode: str) -> dict:
    # Initial retrieval + generation
    retrieved = retrieve(retrieval_query, top_k=top_k, mode=retrieval_mode)
    context = build_context_string(retrieved)
    answer = call_with_retry(lambda: generate_answer(sub_question, context, system_prompt=SYSTEM_PROMPT))
    verification = verify_answer(sub_question, answer, retrieved)

    out = {
        "query": retrieval_query,
        "answer": answer,
        "retrieved": retrieved,
        "verification": verification,
        "intervened": False,
        "rescue_used": False,
        "rescue_attempt": None,
    }

    # Determine intervention eligibility.
    state = verification["state"]
    should_rescue = (
        (state == "UNSUPPORTED" and INTERVENE_UNSUPPORTED) or
        (state == "UNANSWERABLE" and INTERVENE_UNANSWERABLE)
    )

    if not should_rescue:
        return out

    # One-step rescue policy:
    # 1) broaden retrieval with higher k,
    # 2) regenerate with stricter grounding prompt,
    # 3) re-verify and accept rescue only if support improves.
    rescue_query = f"{sub_question} {answer}" if "UNANSWERABLE" not in answer.upper() else sub_question
    rescue_retrieved = retrieve(rescue_query, top_k=RESCUE_TOP_K, mode=retrieval_mode)
    rescue_context = build_context_string(rescue_retrieved)
    rescue_answer = call_with_retry(lambda: generate_answer(sub_question, rescue_context, system_prompt=RESCUE_SYSTEM_PROMPT))
    rescue_ver = verify_answer(sub_question, rescue_answer, rescue_retrieved)

    out["intervened"] = True
    out["rescue_attempt"] = {
        "query": rescue_query,
        "answer": rescue_answer,
        "retrieved": rescue_retrieved,
        "verification": rescue_ver,
    }

    # Accept rescue if it moves to SUPPORTED from non-supported.
    if rescue_ver["state"] == "SUPPORTED" and verification["state"] != "SUPPORTED":
        out["rescue_used"] = True
        out["query"] = rescue_query
        out["answer"] = rescue_answer
        out["retrieved"] = rescue_retrieved
        out["verification"] = rescue_ver

    return out

## 3. Full Evaluation Run (Shared 2,000 subset)

This run uses the same subset and retrieval stack as Pipeline 2.
The only architectural difference is verifier-guided rescue on flagged hops.

Checkpointing is enabled to tolerate runtime interruptions.

In [12]:
# Run Pipeline 3 on shared subset with checkpointing

print("Loading HotpotQA bridge questions...")
dataset = load_dataset("hotpotqa/hotpot_qa", "distractor")
val_data = dataset["validation"].filter(lambda x: x["type"] == "bridge")

examples = [val_data[int(i)] for i in eval_indices_shared]
gold_list = [val_gold_indices[int(i)] for i in eval_indices_shared]
print(f"Eval set loaded: {len(examples):,} examples")

CHECKPOINT_PATH = f"{SAVE_DIR}pipeline3_verifier_guided_checkpoint.json"
if os.path.exists(CHECKPOINT_PATH):
    with open(CHECKPOINT_PATH) as f:
        ckpt = json.load(f)
    p3_results = ckpt["results"]
    start_idx = len(p3_results)
    print(f"Resuming from checkpoint: {start_idx} done, {len(examples)-start_idx} remaining")
else:
    p3_results = []
    start_idx = 0
    print("Starting fresh run")

print(f"Config: mode={BEST_MODE!r}, top_k={BEST_TOP_K}, verifier={BEST_VERIFIER}")

for i, (ex, gold_idxs) in enumerate(zip(examples[start_idx:], gold_list[start_idx:]), start=start_idx):
    question = ex["question"]
    gold = ex["answer"]

    # Reuse saved decomposition from Pipeline 2 for fair comparison.
    q1, q2 = get_decomposition(question)[:2]

    # Hop 1 with possible intervention
    hop1 = run_hop_with_verification(
        sub_question=q1,
        retrieval_query=q1,
        top_k=BEST_TOP_K,
        retrieval_mode=BEST_MODE,
    )

    # Hop 2 retrieval query uses selected hop1 answer unless abstained.
    h1_answer_for_aug = hop1["answer"]
    augmented_q2 = f"{q2} {h1_answer_for_aug}" if "UNANSWERABLE" not in h1_answer_for_aug.upper() else q2

    # Hop 2 with possible intervention
    hop2 = run_hop_with_verification(
        sub_question=q2,
        retrieval_query=augmented_q2,
        top_k=BEST_TOP_K,
        retrieval_mode=BEST_MODE,
    )

    # Final answer generation from merged hop contexts + intermediate findings.
    all_retrieved = list(dict.fromkeys(hop1["retrieved"] + hop2["retrieved"]))[:BEST_TOP_K * 2]
    final_context = build_context_string(all_retrieved)

    final_context_with_reasoning = (
        f"Intermediate findings:\n"
        f"- {q1} -> {hop1['answer']}\n"
        f"- {q2} -> {hop2['answer']}\n\n"
        f"Supporting passages:\n{final_context}"
    )
    predicted = call_with_retry(lambda: generate_answer(question, final_context_with_reasoning, system_prompt=SYSTEM_PROMPT))

    # Score
    em = exact_match(predicted, gold)
    f1 = token_f1(predicted, gold)
    recall_soft = retrieval_recall_at_k(all_retrieved, gold_idxs)
    recall_strict = both_gold_retrieved(all_retrieved, gold_idxs)
    hop1_recall = retrieval_recall_at_k(hop1["retrieved"], gold_idxs)
    hop2_recall = retrieval_recall_at_k(hop2["retrieved"], gold_idxs)
    abstained = int("UNANSWERABLE" in predicted.upper())

    rec = {
        "question": question,
        "gold_answer": gold,
        "predicted_answer": predicted,
        "sub_questions": [q1, q2],

        "hop1_answer": hop1["answer"],
        "hop2_answer": hop2["answer"],
        "retrieved_hop1": hop1["retrieved"],
        "retrieved_hop2": hop2["retrieved"],
        "hop1_verification": hop1["verification"],
        "hop2_verification": hop2["verification"],
        "hop1_ungrounded": int(hop1["verification"]["state"] == "UNSUPPORTED"),
        "hop2_ungrounded": int(hop2["verification"]["state"] == "UNSUPPORTED"),

        "hop1_intervened": int(hop1["intervened"]),
        "hop2_intervened": int(hop2["intervened"]),
        "hop1_rescue_used": int(hop1["rescue_used"]),
        "hop2_rescue_used": int(hop2["rescue_used"]),
        "hop1_rescue_attempt": hop1["rescue_attempt"],
        "hop2_rescue_attempt": hop2["rescue_attempt"],

        "em": em,
        "f1": f1,
        "recall_soft": recall_soft,
        "recall_strict": recall_strict,
        "hop1_recall": hop1_recall,
        "hop2_recall": hop2_recall,
        "abstained": abstained,
    }
    p3_results.append(rec)

    if (i + 1) % 100 == 0:
        with open(CHECKPOINT_PATH, "w") as f:
            json.dump({"results": p3_results, "completed": i + 1}, f)
        print(f"Checkpoint saved: {i+1}/{len(examples)}")

p3_metrics = summarize_results(p3_results)
print("\nPipeline 3 run complete")
for k, v in p3_metrics.items():
    if k != "n":
        print(f"  {k:<25}: {v:.4f}")

# Save
output = {
    "pipeline": "pipeline3_verifier_guided_multihop",
    "split": "shared_eval_2000_seed42",
    "n": len(p3_results),
    "best_mode": BEST_MODE,
    "best_top_k": BEST_TOP_K,
    "embed_model": EMBED_MODEL,
    "llm": "gpt-4o-mini",
    "nli_model": BEST_VERIFIER,
    "intervention": {
        "intervene_unsupported": INTERVENE_UNSUPPORTED,
        "intervene_unanswerable": INTERVENE_UNANSWERABLE,
        "rescue_top_k": RESCUE_TOP_K,
        "verifier_threshold": ENTAILMENT_THRESHOLD,
        "max_passages_verify": MAX_PASSAGES_VERIFY,
    },
    "metrics": p3_metrics,
    "results": p3_results,
}

with open("pipeline3_verifier_guided_results.json", "w") as f:
    json.dump(output, f, indent=2)
with open(f"{SAVE_DIR}pipeline3_verifier_guided_results.json", "w") as f:
    json.dump(output, f, indent=2)

print(f"Saved -> {SAVE_DIR}pipeline3_verifier_guided_results.json")
if os.path.exists(CHECKPOINT_PATH):
    os.remove(CHECKPOINT_PATH)
    print("Checkpoint removed (run complete).")

Loading HotpotQA bridge questions...


README.md:   0%|          | 0.00/9.52k [00:00<?, ?B/s]

distractor/train-00000-of-00002.parquet: reconstructing file:   0%|          |  0.00B /  166MB            

distractor/train-00000-of-00002.parquet: downloading bytes:           |  0.00B            

distractor/train-00001-of-00002.parquet: reconstructing file:   0%|          |  0.00B /  166MB            

...

distractor/validation-00000-of-00001.par(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/90447 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/7405 [00:00<?, ? examples/s]

Filter:   0%|          | 0/7405 [00:00<?, ? examples/s]

Eval set loaded: 2,000 examples
Starting fresh run
Config: mode='hybrid', top_k=10, verifier=cross-encoder/nli-deberta-v3-base
Checkpoint saved: 100/2000
Checkpoint saved: 200/2000
Checkpoint saved: 300/2000
Checkpoint saved: 400/2000
Checkpoint saved: 500/2000
Checkpoint saved: 600/2000
Checkpoint saved: 700/2000
Checkpoint saved: 800/2000
Checkpoint saved: 900/2000
Checkpoint saved: 1000/2000
Checkpoint saved: 1100/2000
Checkpoint saved: 1200/2000
Checkpoint saved: 1300/2000
Checkpoint saved: 1400/2000
Checkpoint saved: 1500/2000
Checkpoint saved: 1600/2000
Checkpoint saved: 1700/2000
Checkpoint saved: 1800/2000
Checkpoint saved: 1900/2000
Checkpoint saved: 2000/2000

Pipeline 3 run complete
  EM                       : 0.4250
  F1                       : 0.5538
  Recall@k (soft)          : 0.9560
  Recall@k (strict)        : 0.6810
  Abstention rate          : 0.2495
  Hop1 recall              : 0.9130
  Hop2 recall              : 0.8700
  Hop1 ungrounded          : 0.2235
  Hop2 un

In [23]:
# FINAL SUMMARY CELL (single-cell replacement for all post-run analysis)
# - Loads P1/P2/P3 outputs from Drive
# - Prints three-way core metrics
# - Prints P2 and P3 decided-only grounding tables (with hop1/hop2 subrows)
# - Prints P3 vs P2 grounding delta summary
# - Keeps UNANSWERABLE separate from UNSUPPORTED

import json
import numpy as np
import math

SAVE_DIR = "/content/drive/MyDrive/datasci266_rag/"

# Choose which P3 file to analyze:
P3_PATH = f"{SAVE_DIR}pipeline3_verifier_guided_results.json"
# P3_PATH = f"{SAVE_DIR}pipeline3_unsupported_only_results.json"
# P3_PATH = f"{SAVE_DIR}pipeline3_unsupported_plus_unanswerable_results.json"

P1_PATH = f"{SAVE_DIR}naive_rag_full_results.json"
P2_PATH = f"{SAVE_DIR}multihop_rag_results.json"
P2_GROUND_PATH = f"{SAVE_DIR}multihop_rag_grounding_results.json"
SHARED_IDX_PATH = f"{SAVE_DIR}eval_indices_shared.npy"

# Helpers
def quick_summarize(results):
    n = len(results)
    return {
        "EM": sum(r["em"] for r in results) / n,
        "F1": sum(r["f1"] for r in results) / n,
        "Recall@k (soft)": sum(r["recall_soft"] for r in results) / n,
        "Recall@k (strict)": sum(r["recall_strict"] for r in results) / n,
        "Abstention rate": sum(r["abstained"] for r in results) / n,
    }

def mean_em(xs):
    return sum(x["em"] for x in xs) / len(xs) if xs else float("nan")

def get_state(r, hop):
    # Preferred source: verifier state
    key = f"{hop}_verification"
    if key in r and isinstance(r[key], dict) and "state" in r[key]:
        return r[key]["state"]
    # Fallback: binary flag (older outputs)
    ukey = f"{hop}_ungrounded"
    if ukey in r:
        return "UNSUPPORTED" if int(r[ukey]) == 1 else "SUPPORTED"
    return None

def summarize_grounding(results):
    rows = []
    for r in results:
        s1 = get_state(r, "hop1")
        s2 = get_state(r, "hop2")
        rows.append({"em": r["em"], "s1": s1, "s2": s2})

    n = len(rows)

    # Overall UNSUPPORTED and UNANSWERABLE rates (separate)
    h1_unsup = [x for x in rows if x["s1"] == "UNSUPPORTED"]
    h2_unsup = [x for x in rows if x["s2"] == "UNSUPPORTED"]
    h1_unans = [x for x in rows if x["s1"] == "UNANSWERABLE"]
    h2_unans = [x for x in rows if x["s2"] == "UNANSWERABLE"]

    # Decided-only subset for conditional EM (exclude UNANSWERABLE hops)
    decided = [x for x in rows if x["s1"] in ("SUPPORTED", "UNSUPPORTED") and x["s2"] in ("SUPPORTED", "UNSUPPORTED")]

    both_sup = [x for x in decided if x["s1"] == "SUPPORTED" and x["s2"] == "SUPPORTED"]
    h1_only  = [x for x in decided if x["s1"] == "UNSUPPORTED" and x["s2"] == "SUPPORTED"]
    h2_only  = [x for x in decided if x["s1"] == "SUPPORTED" and x["s2"] == "UNSUPPORTED"]
    both_uns = [x for x in decided if x["s1"] == "UNSUPPORTED" and x["s2"] == "UNSUPPORTED"]
    one_uns  = h1_only + h2_only

    # Faithfulness failure among correct answers (UNSUPPORTED-based, UNANSWERABLE separate)
    correct = [x for x in rows if x["em"] == 1]
    unfaithful_correct = [x for x in correct if x["s1"] == "UNSUPPORTED" or x["s2"] == "UNSUPPORTED"]

    # Propagation penalty using hop1 decided-only
    hop1_decided = [x for x in rows if x["s1"] in ("SUPPORTED", "UNSUPPORTED")]
    hop1_sup = [x for x in hop1_decided if x["s1"] == "SUPPORTED"]
    hop1_uns = [x for x in hop1_decided if x["s1"] == "UNSUPPORTED"]
    em_h1_sup = mean_em(hop1_sup)
    em_h1_uns = mean_em(hop1_uns)
    prop_penalty = em_h1_sup - em_h1_uns

    return {
        "n": n,
        "decided_n": len(decided),
        "rows": {
            "Both supported": (len(both_sup), mean_em(both_sup)),
            "One unsupported (combined)": (len(one_uns), mean_em(one_uns)),
            "  Hop 1 only unsupported": (len(h1_only), mean_em(h1_only)),
            "  Hop 2 only unsupported": (len(h2_only), mean_em(h2_only)),
            "Both unsupported": (len(both_uns), mean_em(both_uns)),
        },
        "hop1_unsup_rate": len(h1_unsup) / n,
        "hop2_unsup_rate": len(h2_unsup) / n,
        "hop1_unans_rate": len(h1_unans) / n,
        "hop2_unans_rate": len(h2_unans) / n,
        "faithfulness_failure_rate": len(unfaithful_correct) / max(len(correct), 1),
        "propagation_penalty": prop_penalty,
        "em_h1_supported": em_h1_sup,
        "em_h1_unsupported": em_h1_uns,
    }

def arrow(delta, lower_is_better=False, tol=1e-4):
    if abs(delta) <= tol:
        return "─"
    if lower_is_better:
        return "▲" if delta < 0 else "▼"
    return "▲" if delta > 0 else "▼"

# Load data
with open(P1_PATH) as f:
    p1_output = json.load(f)
with open(P2_PATH) as f:
    p2_output = json.load(f)
with open(P2_GROUND_PATH) as f:
    p2_ground = json.load(f)
with open(P3_PATH) as f:
    p3_output = json.load(f)

eval_indices_shared = np.load(SHARED_IDX_PATH)

p1_full_results = p1_output["results"]
p1_subset_results = [p1_full_results[int(i)] for i in eval_indices_shared if int(i) < len(p1_full_results)]

p1_subset_metrics = quick_summarize(p1_subset_results)
p2_metrics = p2_output["metrics"]
p3_metrics = p3_output["metrics"]

p2_results_ground = p2_ground["results"]
p3_results = p3_output["results"]

# 1) Three-way core metrics
print("══ Three-way comparison on shared subset ══")
print(f"{'Metric':<25} {'P1 subset':>12} {'P2':>12} {'P3':>12} {'Δ(P3-P2)':>12}")
print("─" * 80)
for metric in ["EM", "F1", "Recall@k (soft)", "Recall@k (strict)", "Abstention rate"]:
    v1 = p1_subset_metrics[metric]
    v2 = p2_metrics[metric]
    v3 = p3_metrics[metric]
    d = v3 - v2
    a = arrow(d, lower_is_better=(metric == "Abstention rate"))
    print(f"{metric:<25} {v1:>12.4f} {v2:>12.4f} {v3:>12.4f} {a} {d:>+10.4f}")


# 2) Intervention diagnostics
print("\n══ Intervention diagnostics (P3) ══")
print(f"Hop1 intervened: {sum(r.get('hop1_intervened', 0) for r in p3_results) / len(p3_results):.1%}")
print(f"Hop2 intervened: {sum(r.get('hop2_intervened', 0) for r in p3_results) / len(p3_results):.1%}")
print(f"Hop1 rescue used: {sum(r.get('hop1_rescue_used', 0) for r in p3_results) / len(p3_results):.1%}")
print(f"Hop2 rescue used: {sum(r.get('hop2_rescue_used', 0) for r in p3_results) / len(p3_results):.1%}")

# 3) Grounding tables (decided-only)
p2g = summarize_grounding(p2_results_ground)
p3g = summarize_grounding(p3_results)

print("\n══ Grounding state table: Pipeline 2 (decided-only) ══")
print(f"Decided-only n = {p2g['decided_n']} (total n = {p2g['n']})")
print(f"{'State':<30} {'n':>6} {'Mean EM':>10}")
print("─" * 52)
for k, (cnt, em) in p2g["rows"].items():
    print(f"{k:<30} {cnt:>6} {em:>10.4f}")
print(f"\nHop1 UNANSWERABLE rate: {p2g['hop1_unans_rate']:.4f}")
print(f"Hop2 UNANSWERABLE rate: {p2g['hop2_unans_rate']:.4f}")

print("\n══ Grounding state table: Pipeline 3 (decided-only) ══")
print(f"Decided-only n = {p3g['decided_n']} (total n = {p3g['n']})")
print(f"{'State':<30} {'n':>6} {'Mean EM':>10}")
print("─" * 52)
for k, (cnt, em) in p3g["rows"].items():
    print(f"{k:<30} {cnt:>6} {em:>10.4f}")
print(f"\nHop1 UNANSWERABLE rate: {p3g['hop1_unans_rate']:.4f}")
print(f"Hop2 UNANSWERABLE rate: {p3g['hop2_unans_rate']:.4f}")

# 4) P3 vs P2 grounding delta summary
print("\n══ P3 vs P2 grounding summary (delta = P3 - P2) ══")
print(f"{'Metric':<38} {'P2':>10} {'P3':>10} {'Δ':>10}")
print("─" * 76)

rows = [
    ("Hop1 ungrounded rate", p2g["hop1_unsup_rate"], p3g["hop1_unsup_rate"], True),
    ("Hop2 ungrounded rate", p2g["hop2_unsup_rate"], p3g["hop2_unsup_rate"], True),
    ("Faithfulness failure (among EM=1)", p2g["faithfulness_failure_rate"], p3g["faithfulness_failure_rate"], True),
    ("Propagation penalty (EMsup-EMunsup)", p2g["propagation_penalty"], p3g["propagation_penalty"], False),
    ("EM | hop1 supported", p2g["em_h1_supported"], p3g["em_h1_supported"], False),
    ("EM | hop1 unsupported", p2g["em_h1_unsupported"], p3g["em_h1_unsupported"], False),
]

for name, v2, v3, lower_better in rows:
    d = v3 - v2
    a = arrow(d, lower_is_better=lower_better)
    print(f"{name:<38} {v2:>10.4f} {v3:>10.4f} {a} {d:>+8.4f}")

══ Three-way comparison on shared subset ══
Metric                       P1 subset           P2           P3     Δ(P3-P2)
────────────────────────────────────────────────────────────────────────────────
EM                              0.3385       0.4240       0.4250 ▲    +0.0010
F1                              0.4532       0.5552       0.5538 ▼    -0.0013
Recall@k (soft)                 0.9525       0.9570       0.9560 ▼    -0.0010
Recall@k (strict)               0.4610       0.6950       0.6810 ▼    -0.0140
Abstention rate                 0.3470       0.2400       0.2495 ▼    +0.0095

══ Intervention diagnostics (P3) ══
Hop1 intervened: 33.1%
Hop2 intervened: 33.2%
Hop1 rescue used: 10.8%
Hop2 rescue used: 7.8%

══ Grounding state table: Pipeline 2 (decided-only) ══
Decided-only n = 1126 (total n = 2000)
State                               n    Mean EM
────────────────────────────────────────────────────
Both supported                    336     0.5982
One unsupported (combined)     

## 4. Policy Ablation

This optional final section compares two intervention policies under identical settings:

1. **UNSUPPORTED-only intervention** (primary fair-comparison policy)
2. **UNSUPPORTED + UNANSWERABLE intervention** (more aggressive recovery policy)

Both runs use the same shared subset, retriever config, verifier, prompts, and scoring functions.
Each run is checkpointed and saved with a distinct output filename, then summarized side by side.

Use this section to answer whether rescuing `UNANSWERABLE` hops improves end performance or mainly increases intervention cost.

In [ ]:
# Section 1: Storage Linking and Metric Calculation Wrapper
import json
import numpy as np
from google.colab import drive

# Expose global operational policy parameters to enable back-to-back testing.
def run_pipeline3_policy(intervene_unsupported: bool, intervene_unanswerable: bool, run_tag: str):
    global INTERVENE_UNSUPPORTED, INTERVENE_UNANSWERABLE
    INTERVENE_UNSUPPORTED = intervene_unsupported
    INTERVENE_UNANSWERABLE = intervene_unanswerable

    # Load the benchmark evaluation dataset and filter for bridge-type multi-hop questions
    dataset = load_dataset("hotpotqa/hotpot_qa", "distractor")
    val_data = dataset["validation"].filter(lambda x: x["type"] == "bridge")

    # Slice matching test records and ground-truth indexing sets across variants
    examples = [val_data[int(i)] for i in eval_indices_shared]
    gold_list = [val_gold_indices[int(i)] for i in eval_indices_shared]

    # Establish target paths for crash recovery logs and final experiment state outputs
    ckpt_path = f"{SAVE_DIR}pipeline3_{run_tag}_checkpoint.json"
    out_path = f"{SAVE_DIR}pipeline3_{run_tag}_results.json"

    # Crash-Recovery System: Detect an active file to pick up execution where it failed
    if os.path.exists(ckpt_path):
        with open(ckpt_path) as f:
            ckpt = json.load(f)
        results = ckpt["results"]
        start_idx = len(results)
        print(f"[{run_tag}] Resuming {start_idx}/{len(examples)}")
    else:
        results = []
        start_idx = 0
        print(f"[{run_tag}] Starting fresh run ({len(examples)} examples)")


# Section 2: Multi-hop Pipeline Execution Loop with Retry Logic
    for i, (ex, gold_idxs) in enumerate(zip(examples[start_idx:], gold_list[start_idx:]), start=start_idx):
        question = ex["question"]
        gold = ex["answer"]

        # Parse sub-questions to isolate reasoning legs
        q1, q2 = get_decomposition(question)[:2]

        # --- Reasoning Hop 1 ---
        hop1 = run_hop_with_verification(
            sub_question=q1,
            retrieval_query=q1,
            top_k=BEST_TOP_K,
            retrieval_mode=BEST_MODE,
        )

        # Append preceding output to enhance step-2 query generation context
        h1_answer_for_aug = hop1["answer"]
        augmented_q2 = f"{q2} {h1_answer_for_aug}" if "UNANSWERABLE" not in h1_answer_for_aug.upper() else q2

        # --- Reasoning Hop 2 ---
        hop2 = run_hop_with_verification(
            sub_question=q2,
            retrieval_query=augmented_q2,
            top_k=BEST_TOP_K,
            retrieval_mode=BEST_MODE,
        )


# Section 3: Evidence Synthesis and Metric Compilation
        # Flatten and deduplicate retrieved document passage indices
        all_retrieved = list(dict.fromkeys(hop1["retrieved"] + hop2["retrieved"]))[:BEST_TOP_K * 2]
        final_context = build_context_string(all_retrieved)

        # Inject intermediate steps directly into the prompt to provide a trace of reasoning
        final_context_with_reasoning = (
            f"Intermediate findings:\n"
            f"- {q1} -> {hop1['answer']}\n"
            f"- {q2} -> {hop2['answer']}\n\n"
            f"Supporting passages:\n{final_context}"
        )

        # Call the generator with API retry wrappers to handle rate limit blocks
        predicted = call_with_retry(lambda: generate_answer(question, final_context_with_reasoning, system_prompt=SYSTEM_PROMPT))

        # Core evaluation analytics tracking
        em = exact_match(predicted, gold)
        f1 = token_f1(predicted, gold)
        recall_soft = retrieval_recall_at_k(all_retrieved, gold_idxs)
        recall_strict = both_gold_retrieved(all_retrieved, gold_idxs)
        hop1_recall = retrieval_recall_at_k(hop1["retrieved"], gold_idxs)
        hop2_recall = retrieval_recall_at_k(hop2["retrieved"], gold_idxs)
        abstained = int("UNANSWERABLE" in predicted.upper())

        # Construct comprehensive single-record audit log
        rec = {
            "question": question,
            "gold_answer": gold,
            "predicted_answer": predicted,
            "sub_questions": [q1, q2],
            "hop1_answer": hop1["answer"],
            "hop2_answer": hop2["answer"],
            "retrieved_hop1": hop1["retrieved"],
            "retrieved_hop2": hop2["retrieved"],
            "hop1_verification": hop1["verification"],
            "hop2_verification": hop2["verification"],
            "hop1_ungrounded": int(hop1["verification"]["state"] == "UNSUPPORTED"),
            "hop2_ungrounded": int(hop2["verification"]["state"] == "UNSUPPORTED"),
            "hop1_intervened": int(hop1["intervened"]),
            "hop2_intervened": int(hop2["intervened"]),
            "hop1_rescue_used": int(hop1["rescue_used"]),
            "hop2_rescue_used": int(hop2["rescue_used"]),
            "em": em,
            "f1": f1,
            "recall_soft": recall_soft,
            "recall_strict": recall_strict,
            "hop1_recall": hop1_recall,
            "hop2_recall": hop2_recall,
            "abstained": abstained,
        }
        results.append(rec)

        # Write intermediate states to disk at fixed checkpoint steps to prevent data loss
        if (i + 1) % 100 == 0:
            with open(ckpt_path, "w") as f:
                json.dump({"results": results, "completed": i + 1}, f)
            print(f"[{run_tag}] Checkpoint: {i+1}/{len(examples)}")


# Section 4: Metrics Packaging and Cleanup
    metrics = summarize_results(results)
    output = {
        "pipeline": "pipeline3_verifier_guided_multihop",
        "variant": run_tag,
        "split": "shared_eval_2000_seed42",
        "n": len(results),
        "best_mode": BEST_MODE,
        "best_top_k": BEST_TOP_K,
        "embed_model": EMBED_MODEL,
        "llm": "gpt-4o-mini",
        "nli_model": BEST_VERIFIER,
        "intervention": {
            "intervene_unsupported": intervene_unsupported,
            "intervene_unanswerable": intervene_unanswerable,
            "rescue_top_k": RESCUE_TOP_K,
            "verifier_threshold": ENTAILMENT_THRESHOLD,
            "max_passages_verify": MAX_PASSAGES_VERIFY,
        },
        "metrics": metrics,
        "results": results,
    }

    # Save final structured outputs and remove the temporary recovery checkpoint file
    with open(out_path, "w") as f:
        json.dump(output, f, indent=2)

    if os.path.exists(ckpt_path):
        os.remove(ckpt_path)

    return output


# Section 5: Ablation Execution and Performance Contrast
# Run Variant A: Intervene only when an answer cannot find contextual grounding
out_a = run_pipeline3_policy(intervene_unsupported=True, intervene_unanswerable=False, run_tag="unsupported_only")

# Run Variant B: Aggressive Strategy - Intervene on lack of grounding AND explicit unanswerable drops
out_b = run_pipeline3_policy(intervene_unsupported=True, intervene_unanswerable=True, run_tag="unsupported_plus_unanswerable")


def rate(results, key):
    """Helper to average specific indicator metrics across historical logs."""
    return sum(r[key] for r in results) / max(len(results), 1)

# Format summary tables comparing baseline variants side-by-side
print("\n══ Policy Ablation Summary ══")
print(f"{'Metric':<25} {'Unsupported only':>18} {'Unsupported+Unans':>20} {'Delta':>10}")
print("─" * 80)
for metric in ["EM", "F1", "Recall@k (soft)", "Recall@k (strict)", "Abstention rate", "Hop1 ungrounded", "Hop2 ungrounded"]:
    a = out_a["metrics"][metric]
    b = out_b["metrics"][metric]
    d = b - a
    # Mark significant variance changes (> 0.1% margin) with explicit direction markers
    arrow = "▲" if d > 0.001 else ("▼" if d < -0.001 else "─")
    print(f"{metric:<25} {a:>18.4f} {b:>20.4f} {arrow} {d:>+8.4f}")

# Extract active trigger interaction measurements
a_h1_int = rate(out_a["results"], "hop1_intervened")
a_h2_int = rate(out_a["results"], "hop2_intervened")
b_h1_int = rate(out_b["results"], "hop1_intervened")
b_h2_int = rate(out_b["results"], "hop2_intervened")

print()
print("Intervention rates")
print(f"  Hop1 intervened: {a_h1_int:.1%} -> {b_h1_int:.1%}")
print(f"  Hop2 intervened: {a_h2_int:.1%} -> {b_h2_int:.1%}")
print()
print(f"Saved variant outputs to:\n  {SAVE_DIR}pipeline3_unsupported_only_results.json\n  {SAVE_DIR}pipeline3_unsupported_plus_unanswerable_results.json")